This notebook process one-by-one the netcdf files in `data/SWOT_L3_LR_SSH_Expert_v3_0`.

For each file it:
- checks if the corresponding file is already present in the directory `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion`,
- if it is not present, it:
    - estimates the cyclogeostrophic currents using the gradient-wind, fixed-point, and minimization-based approaches,
    - creates a new dataset from the estimated currents components,
    - stores it in the directory `data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion` using the original filename.

In [1]:
import glob
import os
import time
import warnings

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import optax
import xarray as xr

from jaxparrow.cyclogeostrophy import fixed_point, gradient_wind, minimization_based

In [ ]:
warnings.filterwarnings("ignore")

INPUT_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_err"
OUTPUT_DIR = "data/SWOT_L3_LR_SSH_Expert_v3_0_cg_inversion"

while True:
    for filepath in sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc"))):
        filename = os.path.basename(filepath)
        output_path = os.path.join(OUTPUT_DIR, filename)

        if os.path.exists(output_path):
            try:
                ds_out = xr.open_dataset(output_path)
                ds_out.ucg_mb.load()
                ds_out.vcg_mb.load()
                ds_out.close()
                continue
            except Exception:
                pass

        try:
            ds = xr.open_dataset(filepath)

            ds = ds.where(np.abs(ds.latitude) > 10)

            ug = ds.ugos_filtered.values
            vg = ds.vgos_filtered.values
            lat = ds.latitude.values
            lon = ds.longitude.values
            
            res_fp = fixed_point(lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg, is_grid_rectilinear=False)

            res_gw = gradient_wind(lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg, is_grid_rectilinear=False)

            res_mb = minimization_based(
                lat_t=lat, lon_t=lon, ug_t=ug, vg_t=vg, 
                is_grid_rectilinear=False,
                optim=optax.chain(optax.clip(1), optax.sgd(learning_rate=5e-3))
            )

            ds_out = xr.Dataset(
                {
                    "ucg_fp": xr.DataArray(res_fp.ucg, dims=ds.ugos_filtered.dims, coords=ds.ugos_filtered.coords),
                    "vcg_fp": xr.DataArray(res_fp.vcg, dims=ds.vgos_filtered.dims, coords=ds.vgos_filtered.coords),
                    "ucg_gw": xr.DataArray(res_gw.ucg, dims=ds.ugos_filtered.dims, coords=ds.ugos_filtered.coords),
                    "vcg_gw": xr.DataArray(res_gw.vcg, dims=ds.vgos_filtered.dims, coords=ds.vgos_filtered.coords),
                    "ucg_mb": xr.DataArray(res_mb.ucg, dims=ds.ugos_filtered.dims, coords=ds.ugos_filtered.coords),
                    "vcg_mb": xr.DataArray(res_mb.vcg, dims=ds.vgos_filtered.dims, coords=ds.vgos_filtered.coords),
                },
            )

            ds_out.to_netcdf(output_path)

            ds.close()
            ds_out.close()
        except Exception:
            print(f"Error for file {filename}")
    
    time.sleep(1)

: 

: 

: 